# 💼 05 — Business Insights Dashboard
**E-Commerce Customer & Revenue Analytics Platform**

This notebook presents the complete business intelligence summary:
- All KPI metrics in formatted tables
- Executive summary cards
- Key findings from EDA, segmentation, and forecasting
- Strategic recommendations
- Full pipeline runner (runs all analyses end-to-end)

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.join('..', 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

BASE = '..'
CLEAN_PATH   = os.path.join(BASE, 'data', 'processed', 'cleaned_data.csv')
KPI_PATH     = os.path.join(BASE, 'data', 'processed', 'kpis.json')
RFM_PATH     = os.path.join(BASE, 'data', 'processed', 'rfm_segments.csv')
METRICS_PATH = os.path.join(BASE, 'data', 'processed', 'forecast_metrics.json')
FORECAST_PATH= os.path.join(BASE, 'data', 'processed', 'forecast_results.csv')

df = pd.read_csv(CLEAN_PATH, parse_dates=['InvoiceDate'])
print('Data loaded.')

## Run Full Pipeline (if first time)

In [ ]:
import logging
logging.basicConfig(level=logging.WARNING)

# Only run if KPIs haven't been computed yet
if not os.path.exists(KPI_PATH):
    print('Running KPI analysis...')
    from kpi_analysis import run_kpi_analysis
    run_kpi_analysis(df)

if not os.path.exists(RFM_PATH):
    print('Running RFM segmentation...')
    from customer_segmentation import run_segmentation
    run_segmentation(df)

if not os.path.exists(METRICS_PATH):
    print('Running forecasting...')
    from forecasting import run_forecasting
    run_forecasting(df)

print('All analyses ready.')

## Section 1: Executive KPI Dashboard

In [ ]:
with open(KPI_PATH) as f:
    kpis = json.load(f)

kpi_display = [
    ('Total Revenue',          f"£{kpis['total_revenue']:,.2f}",         '#6C63FF'),
    ('Total Orders',           f"{kpis['total_orders']:,}",               '#43B6C8'),
    ('Total Customers',        f"{kpis['total_customers']:,}",            '#56D79E'),
    ('Avg Order Value',        f"£{kpis['avg_order_value']:,.2f}",        '#FFB347'),
    ('Revenue per Customer',   f"£{kpis['revenue_per_customer']:,.2f}",   '#6C63FF'),
    ('Repeat Purchase Rate',   f"{kpis['repeat_purchase_rate']:.1f}%",    '#43B6C8'),
    ('Customer Retention',     f"{kpis['customer_retention_rate']:.1f}%", '#56D79E'),
    ('Avg Monthly Growth',     f"{kpis['avg_monthly_growth_pct']:.1f}%",  '#FF6B6B'),
]

fig = plt.figure(figsize=(16, 4))
fig.patch.set_facecolor('#1a1a2e')
gs = gridspec.GridSpec(1, 8, figure=fig, wspace=0.05)

for i, (label, value, color) in enumerate(kpi_display):
    ax = fig.add_subplot(gs[0, i])
    ax.set_facecolor('#16213e')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')
    ax.text(0.5, 0.72, value, ha='center', va='center', fontsize=13,
            fontweight='bold', color=color, transform=ax.transAxes)
    ax.text(0.5, 0.28, label, ha='center', va='center', fontsize=8,
            color='#aaaacc', transform=ax.transAxes, wrap=True)
    for sp in ax.spines.values():
        sp.set_edgecolor(color)
        sp.set_linewidth(2)

fig.suptitle('E-Commerce KPI Dashboard', fontsize=15, fontweight='bold',
             color='white', y=1.04)
plt.tight_layout()
plt.show()

## Section 2: Customer Segment Summary

In [ ]:
rfm = pd.read_csv(RFM_PATH)

seg_summary = rfm.groupby('Segment').agg(
    Customers    = ('CustomerID', 'count'),
    AvgRecency   = ('Recency', 'mean'),
    AvgFrequency = ('Frequency', 'mean'),
    TotalRevenue = ('Monetary', 'sum'),
    AvgRevenue   = ('Monetary', 'mean'),
).reset_index()

seg_summary['Revenue Share %'] = (seg_summary['TotalRevenue'] / seg_summary['TotalRevenue'].sum() * 100).round(1)
seg_summary['% of Customers']  = (seg_summary['Customers'] / seg_summary['Customers'].sum() * 100).round(1)
seg_summary = seg_summary.sort_values('TotalRevenue', ascending=False)

for col in ['AvgRecency', 'AvgFrequency', 'TotalRevenue', 'AvgRevenue']:
    seg_summary[col] = seg_summary[col].round(1)

print('=== Customer Segment Summary ===')
seg_summary

## Section 3: Top 10 Customers by Lifetime Revenue

In [ ]:
top_customers = (df.groupby('CustomerID').agg(
    TotalRevenue = ('Revenue', 'sum'),
    TotalOrders  = ('InvoiceNo', 'nunique'),
    Country      = ('Country', lambda x: x.mode()[0])
).reset_index()
 .sort_values('TotalRevenue', ascending=False)
 .head(10)
 .round(2))

print('=== Top 10 Customers by Lifetime Revenue ===')
top_customers

## Section 4: Top 10 Products

In [ ]:
top_products = (df.groupby(['StockCode', 'Description']).agg(
    TotalRevenue = ('Revenue', 'sum'),
    UnitsSold    = ('Quantity', 'sum'),
    AvgPrice     = ('UnitPrice', 'mean')
).reset_index()
 .sort_values('TotalRevenue', ascending=False)
 .head(10)
 .round(2))

print('=== Top 10 Products ===')
top_products

## Section 5: Forecast Summary

In [ ]:
with open(METRICS_PATH) as f:
    metrics = json.load(f)

forecast = pd.read_csv(FORECAST_PATH)

print('=== Forecast Model Metrics ===')
for k, v in metrics.items():
    print(f'  {k:<6}: {v}')

print(f'\n=== 6-Month Revenue Forecast ===')
for _, row in forecast.iterrows():
    print(f'  {row["YearMonth"]}  →  £{row["ForecastRevenue"]:>12,.2f}')

print(f'\n  Total Projected: £{forecast["ForecastRevenue"].sum():,.2f}')

## Section 6: Generate Business Report

In [ ]:
from report_generator import run_report_generator
report_path = run_report_generator()
print(f'Report saved: {report_path}')

# Display first 50 lines
with open(report_path, 'r', encoding='utf-8') as f:
    lines = f.readlines()
print(''.join(lines[:50]))

## ✅ Project Complete!

All analyses have been run successfully. Output files:

| File | Description |
|------|-------------|
| `data/processed/cleaned_data.csv` | Clean transaction data |
| `data/processed/customer_summary.csv` | Customer-level aggregates |
| `data/processed/rfm_segments.csv` | RFM customer segments |
| `data/processed/forecast_results.csv` | 6-month revenue forecast |
| `data/processed/ecommerce.db` | SQLite database |
| `visualizations/*.png` | All charts (11 files) |
| `dashboard/dashboard_dataset.csv` | Power BI-ready export |
| `reports/business_report.md` | Populated business report |
